# Retrain the 5-model ensemble with C2 keep-rows

Same pipeline as `colab_retrain_2026-09.ipynb`, with one architectural change:
`model.stft_keep_rows` is turned **on**, so the STFT branch keeps the frequency
rows instead of averaging them away with `f.mean(dim=2)`.

**Why.** On a single model this lifted FHSS recall with a jammer in the window
from 55.4% to 73.9%, a gain of 18.5 points, while JAMMING precision improved to
99.5% and the civilian false alarm rate fell to 0.015%. All ten pre-registered
checks passed. It also cost about 4 points on FHSS when the other military
emitter is present, which is the open question this run settles.

**What comes out:** `results/ensemble_0..4.pt`, `results/best_model.pt`, a
recalibrated evaluation, and the two probes.

Every path in this notebook is unchanged from the original. Only the branch,
the config flag, and the evaluation cells are different.

**Runtime, Change runtime type, GPU** before starting.

## 1. Get the code

The C2 code lives on `c2-keep-stft-rows`, not on `main`. That branch also
carries the fftshift fix the keep-rows path depends on, so the frequency axis
is monotonic before the learned layer sees it.

In [ ]:
BRANCH = 'c2-keep-stft-rows'

%cd /content
!rm -rf sedicAI_NEXA
!git clone -q -b $BRANCH https://github.com/eavan127/sedicAI_NEXA.git
%cd /content/sedicAI_NEXA
!git log --oneline -1

In [ ]:
# Colab already has torch, numpy, scipy, sklearn, matplotlib.
!pip install -q pyyaml h5py

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else 'CPU ONLY - stop and switch the runtime to GPU')

## 2. Turn C2 on

`scripts/train_ensemble.py` reads `configs/default.yaml` directly and has **no
command line flag** for this, unlike the probes and the evaluator. The file
itself has to say `true` before training starts, or the run silently retrains
the baseline architecture and the GPU hours are wasted.

This cell must run **before anything imports `src.config`**, because the config
is read once and cached.

In [ ]:
import pathlib, re

p = pathlib.Path('configs/default.yaml')
text = p.read_text()
patched = re.sub(r'^(\s*stft_keep_rows:\s*)false\s*$', r'\1true', text, flags=re.M)
assert patched != text, ('stft_keep_rows: false not found in configs/default.yaml. '
                         'Check the branch is c2-keep-stft-rows.')
p.write_text(patched)

for line in patched.splitlines():
    if 'stft_keep_rows' in line and not line.strip().startswith('#'):
        print('config now says:', line.strip())

## 3. Get the data in

Unchanged from the original notebook. The only addition is that the copy looks
in `sedic/` first and then `sedic/eavan-retrain/`, so it works whichever of the
two holds the arrays. Nothing is moved and no new folder is created in Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pathlib, shutil

NEEDED = ['X.npy', 'y.npy', 'snr_labels.npy']
CANDIDATES = [
    pathlib.Path('/content/drive/MyDrive/sedic'),
    pathlib.Path('/content/drive/MyDrive/sedic/eavan-retrain'),
]

source = next((d for d in CANDIDATES if all((d / n).is_file() for n in NEEDED)), None)
assert source is not None, ('none of these Drive folders hold all three arrays:\n  '
                            + '\n  '.join(str(d) for d in CANDIDATES))
print('source:', source, '\n')

dest = pathlib.Path('data/processed')
dest.mkdir(parents=True, exist_ok=True)
for name in NEEDED:
    if not (dest / name).is_file():
        shutil.copy(source / name, dest / name)
    print(f'  {name:<16}{(dest / name).stat().st_size:>13,} bytes')

## 4. Verify the data BEFORE spending GPU time

Checks the shape, the multi-hot label format, and that the SNR bins match the
config. Seconds to run.

**If any assert fires, stop.** Do not run the training cells.

In [ ]:
import numpy as np
from src.config import CFG, CLASSES

X = np.load('data/processed/X.npy', mmap_mode='r')
y = np.load('data/processed/y.npy')
snr = np.load('data/processed/snr_labels.npy')

print('X  ', X.shape, X.dtype)
print('y  ', y.shape, y.dtype)
print('snr', snr.shape)

assert X.ndim == 3 and X.shape[1] == 2, f'expected (N, 2, L), got {X.shape}'
assert y.ndim == 2 and y.shape[1] == len(CLASSES), (
    f'y must be multi-hot (N, {len(CLASSES)}), got {y.shape}.')
assert len(X) == len(y) == len(snr), 'X / y / snr lengths disagree'

actual = sorted(int(v) for v in set(snr.tolist()))
expected = sorted(CFG['snr_bins_db'])
print('\nSNR bins  config:', expected)
print('SNR bins  actual:', actual)
assert actual == expected, 'SNR bins do NOT match the config. Rebuild locally and re-upload.'

n = y.sum(axis=1)
ni = CLASSES.index('NOISE_FLOOR')
assert (n == 0).sum() == 0, 'some windows carry no label at all'
assert ((y[:, ni] > 0.5) & (n > 1)).sum() == 0, 'NOISE_FLOOR co-occurs with another class'

print('\nper SNR bin :', {b: int((snr == b).sum()) for b in expected})
print('standalone  :', int((n == 1).sum()))
print('composite   :', int((n > 1).sum()))
print('\nOK - data matches the config.')

### Confirm the architecture actually changed

The single number that proves the flag took effect. Baseline is **148,938**
parameters, C2 keep-rows is **181,898**. If this prints the baseline figure,
the config edit did not land and the run below is pointless.

In [ ]:
from src.models.amc_cnn import AMC_CNN

model = AMC_CNN(num_classes=len(CLASSES), input_len=CFG['signal']['window_len'])
n_params = sum(p.numel() for p in model.parameters())

print('stft_keep_rows    =', CFG['model']['stft_keep_rows'])
print('stft_freq_summary =', CFG['model'].get('stft_freq_summary', False))
print('parameters        =', f'{n_params:,}')

assert CFG['model']['stft_keep_rows'] is True, 'flag did not take effect, re-run the config cell'
assert n_params == 181898, f'expected 181,898 parameters for keep-rows, got {n_params:,}'
print('\nOK - training will use the C2 keep-rows architecture')

del model

In [ ]:
# Code sanity check. Catches a bad clone or a missing dependency before the long run.
!python -m pytest -q

## 5. Train the ensemble

Five seeds, sigmoid outputs averaged. Writes `results/ensemble_0.pt` through
`ensemble_4.pt`.

Watch the per-member lines it prints. Those five recalls **are** the seed
variance measurement, which is why `measure_variance.py` stays commented out
below. Running both would pay twice for the same information.

Long cell. Colab disconnects idle tabs, so leave it open and visible.

### Before you start: this is a long run

A single member takes roughly **2.6 hours** on a T4, so five members is about
**13 hours**. Colab will disconnect before that on most runtimes.

`train_ensemble.py` saves each member the moment it finishes, so a disconnect
after member 3 leaves `ensemble_0.pt`, `ensemble_1.pt` and `ensemble_2.pt`
intact in `results/`. But `results/` dies with the runtime, so the cell below
mirrors them to Drive every five minutes. Run it, then start training.

If the session does drop, re-run the setup cells, copy the finished members
back from Drive into `results/`, and train only the ones still missing.

In [ ]:
# Mirror finished checkpoints to Drive every 5 minutes, so a disconnect
# costs at most 5 minutes rather than the whole run. Same Drive folder the
# final copy uses. Daemon thread: it dies with the runtime, nothing to stop.
import pathlib, shutil, threading, time

DRIVE_OUT = pathlib.Path('/content/drive/MyDrive/sedic/results_new')
DRIVE_OUT.mkdir(parents=True, exist_ok=True)


def _autosave():
    while True:
        time.sleep(300)
        for f in pathlib.Path('/content/sedicAI_NEXA/results').glob('*.pt'):
            try:
                shutil.copy(f, DRIVE_OUT / f.name)
            except Exception as e:
                print('autosave skipped', f.name, e)


threading.Thread(target=_autosave, daemon=True).start()
print('autosaving results/*.pt to', DRIVE_OUT, 'every 5 minutes')

In [ ]:
!python scripts/train_ensemble.py --models 5

In [ ]:
# Single-model baseline -> results/best_model.pt.
# Not used by the ensemble scorecard, but the app's Model dropdown offers it
# and the static build exports it.
!python -m src.train

### Optional: extra variance runs

Skip this. The per-member recalls printed above already give the seed spread,
and this roughly doubles the GPU time of the notebook.

In [ ]:
# !python scripts/measure_variance.py --runs 5

## 6. Evaluate

**Read this before reading the numbers.** `train_ensemble.py` and `src.evaluate`
both score at the thresholds in `configs/default.yaml`, which were calibrated
against the **old** checkpoints. The keep-rows model wants roughly
`FHSS 0.15, LFM_RADAR 0.26, JAMMING 0.92` instead of `0.25 / 0.22 / 0.87`, so
its probability distribution has moved.

Scoring a 0.15-shaped model at 0.25 understates FHSS recall. The next two cells
are included for continuity with the old notebook, not as the result. Judge the
run on the recalibrated evaluation after them.

In [ ]:
!python -m src.evaluate

In [ ]:
import json, pathlib

for name, path in [('Single model', 'evals/scorecard.json'),
                   ('Ensemble', 'evals/ensemble_scorecard.json')]:
    p = pathlib.Path(path)
    if not p.is_file():
        print(f'--- {name}: {path} not written ---\n')
        continue
    sc = json.loads(p.read_text())
    print(f'--- {name} ({path}) ---')
    if 'members' in sc:
        print('  per-member recalls (this is the seed spread):')
        for i, m in enumerate(sc['members']):
            print(f'    member {i}: ' + '  '.join(f'{c}={v:.4f}' for c, v in m.items()))
        for c in sc.get('ensemble', {}):
            vals = [m[c] for m in sc['members']]
            print(f'    {c:<12} spread {min(vals):.4f} to {max(vals):.4f} '
                  f'= {100 * (max(vals) - min(vals)):.1f} points')
        print()
    bench = sc.get('benchmark', {})
    for cls, r in bench.get('judged_classes', {}).items():
        print(f"  {cls:<12} recall={r['recall']:.4f}  {'PASS' if r['passed'] else 'FAIL'}")
    if bench:
        print(f"  OVERALL: {'PASS' if bench.get('passed') else 'FAIL'}")
    print()

### The evaluation that counts

This recalibrates thresholds on the validation split, scores the test split,
and prints the side-by-side against the C2 baseline plus the pre-registered
verdict.

One caveat when reading the deltas: `c2_baseline_eval.json` is a **single
model** (`baseline_member0`, 148,938 parameters). This run is a **five-model
ensemble**. Part of every improvement below is ensembling rather than the
architecture, so treat the deltas as indicative and compare like with like
using the single-member probe cells further down.

Writes `results/eval_rows_ens.json` and `results/thresholds_rows_ens.json`.

In [ ]:
!python scripts/evaluate_experiment.py   --checkpoint results/ensemble_0.pt   --checkpoint results/ensemble_1.pt   --checkpoint results/ensemble_2.pt   --checkpoint results/ensemble_3.pt   --checkpoint results/ensemble_4.pt   --stft-keep-rows   --name rows_ens --baseline docs/experiments/c2_baseline_eval.json

## 7. Probes

Two things the test split alone cannot answer.

`high_snr_probe.py` separates standalone windows from FHSS-plus-radar and
FHSS-plus-jammer, so the high-SNR decline can be attributed to one of them.
`probe_jsr.py` sweeps jammer-to-signal ratio at a fixed 10 dB SNR.

Both are inference only, so they take minutes rather than hours. Both run
twice: once on the full ensemble, which is the number you would quote, and once
on a single member, which is the only fair comparison against the single-model
baselines in `docs/experiments/`.

In [ ]:
# Ensemble. This is the headline number.
!python scripts/high_snr_probe.py --n 300 --class FHSS   --ensemble --n-models 5 --stft-keep-rows   --thresholds results/thresholds_rows_ens.json --out results/high_snr_rows_ens.json

In [ ]:
# Single member, like-for-like against docs/experiments/c2_baseline_high_snr.json.
!python scripts/high_snr_probe.py --n 300 --class FHSS   --checkpoint results/ensemble_0.pt --stft-keep-rows   --thresholds results/thresholds_rows_ens.json --out results/high_snr_rows_1model.json

In [ ]:
# JSR sweep at fixed 10 dB SNR. Baseline FHSS recall at +10 dB JSR was 0.048.
!python scripts/probe_jsr.py --n 600 --members 5 --stft-keep-rows   --thresholds results/thresholds_rows_ens.json --out results/jsr_rows_ens.json

!python scripts/probe_jsr.py --n 600 --members 1 --stft-keep-rows   --thresholds results/thresholds_rows_ens.json --out results/jsr_rows_1model.json

### Pre-registered verdict

Judges the single-member probes against the single-member baselines, which is
the comparison the checks in `docs/POST_STAGE1_FIXES.md` E5 were written for.
Exits non-zero if any check fails.

In [ ]:
!python scripts/evaluate_experiment.py --verdict-only   --eval-json results/eval_rows_ens.json   --baseline docs/experiments/c2_baseline_eval.json   --high-snr results/high_snr_rows_1model.json   --baseline-high-snr docs/experiments/c2_baseline_high_snr.json   --jsr results/jsr_rows_1model.json --baseline-jsr docs/experiments/c2_baseline_jsr.json

In [ ]:
from IPython.display import Image, display
display(Image('evals/confusion_matrix.png'))
display(Image('evals/accuracy_vs_snr.png'))

## 8. Get the results out - DO NOT SKIP

Colab deletes everything when the runtime ends.

**Check what is already in that folder first.** The destination is unchanged
from the original notebook, so if a previous run left checkpoints there they
will be overwritten. The next cell lists the folder before anything is copied,
so you can see what is at risk. Your local `calibrated-result` copy is
untouched either way.

In [ ]:
!echo '--- what is already there ---'
!ls -la /content/drive/MyDrive/sedic/results_new/ 2>/dev/null || echo '(folder does not exist yet)'

In [ ]:
!mkdir -p /content/drive/MyDrive/sedic/results_new
!cp results/ensemble_*.pt  /content/drive/MyDrive/sedic/results_new/
!cp results/best_model.pt  /content/drive/MyDrive/sedic/results_new/
!cp results/eval_rows_ens.json results/thresholds_rows_ens.json /content/drive/MyDrive/sedic/results_new/
!cp results/high_snr_rows_ens.json results/high_snr_rows_1model.json /content/drive/MyDrive/sedic/results_new/
!cp results/jsr_rows_ens.json results/jsr_rows_1model.json /content/drive/MyDrive/sedic/results_new/
!cp -r evals               /content/drive/MyDrive/sedic/results_new/
!ls -la /content/drive/MyDrive/sedic/results_new/

## 9. What to read, in order

1. **Per-member spread**, from the scorecard cell. If FHSS recall swings more
   than about 3 points across the five seeds, the 4 point drop on
   FHSS-with-emitter seen in the single-model run is inside noise and should
   not be written up as a finding. If the five land within a point of each
   other, it is real and belongs in the limitations section.

2. **The verdict block.** Ten checks. The one that matters is FHSS recall with
   a jammer present, which needs to be at least 10 points above baseline.

3. **The JSR sweep at +10 dB.** Baseline was 0.048. Anything above 0.25 means
   keeping the rows recovered real detection under a strong jammer. Check
   `jammer_called_fhss` has not climbed much above the baseline 0.017, since a
   variant that finds FHSS by labelling jammers as FHSS is not a fix.

4. **The overlay row of the high-SNR probe.** On the single model it fell 30
   points from its peak to +10 dB. Less than that is progress, and it will
   probably not reach zero, because the remaining cause is that the model has
   frequency position but no feature for frequency change over time.

Thresholds are already calibrated by section 6, so
`results/thresholds_rows_ens.json` is what goes into
`multilabel_thresholds_per_class` if you adopt this run. Do not reuse the old
values.

Keep `calibrated-result` as the fallback submission until these numbers are in
and compared.